<a href="https://colab.research.google.com/github/MagdyTarek18/Week1-Repo/blob/main/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MagdyTarek18/Week1-Repo/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

This playbook ranks content for **human review**.

The goal is not to automatically change content. It is to help an editor decide what deserves attention first.

I use simple, transparent signals:

- **Visible:** at least 300 impressions
- **Stale:** at least 180 days since last update
- **Low CTR opportunity:** page-one position with CTR below 1%
- **Striking distance:** average position 11–20
- **Thin but visible:** fewer than 2,000 words with meaningful impressions

Each page receives one reason code and one suggested action.

For cost/value thinking, impressions are used only as a **visibility/value proxy**, not as money or revenue. Actions also receive a simple relative effort cost. The priority score favors visible opportunities that appear cheaper to review.

The FlyRank research paper observed a content decay pattern after the stronger early-life period and noted that older refreshed pages can perform differently from stale ones. I use staleness only as a review cue, not as proof that refreshing a page will improve it.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display



candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("content_refresh_anonymized.csv")
]

DATA_PATH = next(
    (p for p in candidates if p.exists()),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find content_refresh_anonymized.csv"
    )

df = pd.read_csv(DATA_PATH)

# Find repo root for later exports
if (
    DATA_PATH.parent.name == "raw"
    and DATA_PATH.parent.parent.name == "data"
):
    REPO_ROOT = DATA_PATH.parent.parent.parent
else:
    REPO_ROOT = Path.cwd()




visible = df["impressions_90d"] >= 300

stale = (
    visible &
    (df["days_since_last_update"] >= 180)
)

page_one_low_ctr = (
    visible &
    (df["avg_position"] > 0) &
    (df["avg_position"] <= 10) &
    (df["ctr"] < 1.0)
)

striking_distance = (
    visible &
    (df["avg_position"] > 10) &
    (df["avg_position"] <= 20)
)

thin_visible = (
    visible &
    df["word_count"].notna() &
    (df["word_count"] < 2000)
)




df["reason_code"] = np.select(
    [
        stale & page_one_low_ctr,
        stale,
        page_one_low_ctr,
        striking_distance,
        thin_visible
    ],
    [
        "stale_and_low_ctr",
        "stale_but_visible",
        "page_one_low_ctr",
        "striking_distance",
        "thin_but_visible"
    ],
    default="monitor_only"
)




action_map = {
    "stale_and_low_ctr": (
        "Stale + CTR opportunity",
        "REFRESH_AND_CTR_REVIEW",
        2
    ),

    "stale_but_visible": (
        "Stale but visible",
        "REFRESH_REVIEW",
        2
    ),

    "page_one_low_ctr": (
        "Page-one CTR opportunity",
        "CTR_REVIEW",
        1
    ),

    "striking_distance": (
        "Near page one",
        "RELEVANCE_REVIEW",
        2
    ),

    "thin_but_visible": (
        "Thin but visible",
        "DEPTH_REVIEW",
        3
    ),

    "monitor_only": (
        "Monitor",
        "NO_ACTION",
        99
    )
}

df["archetype"] = df["reason_code"].map(
    lambda x: action_map[x][0]
)

df["action_label"] = df["reason_code"].map(
    lambda x: action_map[x][1]
)

df["cost_weight"] = df["reason_code"].map(
    lambda x: action_map[x][2]
)



df["value_proxy"] = np.log1p(
    df["impressions_90d"]
)

df["priority_score"] = np.where(
    df["action_label"] != "NO_ACTION",
    df["value_proxy"] / df["cost_weight"],
    0
)



queue = (
    df[df["action_label"] != "NO_ACTION"]
    .sort_values(
        ["priority_score", "impressions_90d"],
        ascending=False
    )
    .copy()
)

queue["rank"] = range(
    1,
    len(queue) + 1
)


# Mapping table
mapping_df = pd.DataFrame(
    [
        {
            "reason_code": reason,
            "archetype": values[0],
            "action": values[1],
            "relative_cost": values[2]
        }
        for reason, values in action_map.items()
        if reason != "monitor_only"
    ]
)

print("Archetype -> action mapping:")
display(mapping_df)

print("\nTop 10 ranked actions:")
display(
    queue[
        [
            "rank",
            "content_id",
            "priority_score",
            "archetype",
            "reason_code",
            "action_label",
            "impressions_90d",
            "avg_position",
            "ctr",
            "days_since_last_update"
        ]
    ].head(10)
)

Archetype -> action mapping:


,reason_code,archetype,action,relative_cost
0,stale_and_low_ctr,Stale + CTR opportunity,REFRESH_AND_CTR_REVIEW,2
1,stale_but_visible,Stale but visible,REFRESH_REVIEW,2
2,page_one_low_ctr,Page-one CTR opportunity,CTR_REVIEW,1
3,striking_distance,Near page one,RELEVANCE_REVIEW,2
4,thin_but_visible,Thin but visible,DEPTH_REVIEW,3



Top 10 ranked actions:


,rank,content_id,priority_score,archetype,reason_code,action_label,impressions_90d,avg_position,ctr,days_since_last_update
6653,1,content_5fe46e04994d,13.157182,Page-one CTR opportunity,page_one_low_ctr,CTR_REVIEW,517715,4.2,0.14,104
17812,2,content_aaef01a50def,13.156011,Page-one CTR opportunity,page_one_low_ctr,CTR_REVIEW,517109,5.4,0.25,22
26844,3,content_8c19996aa890,13.140700,Page-one CTR opportunity,page_one_low_ctr,CTR_REVIEW,509252,2.5,0.15,20
21819,4,content_4c36c775b818,13.045707,Page-one CTR opportunity,page_one_low_ctr,CTR_REVIEW,463103,2.3,0.41,20
29879,5,content_1a9e894be2e2,12.938876,Page-one CTR opportunity,page_one_low_ctr,CTR_REVIEW,416180,4.0,0.23,22
13537,6,content_2c2606c5d176,12.758232,Page-one CTR opportunity,page_one_low_ctr,CTR_REVIEW,347399,4.2,0.53,104
18870,7,content_db5989a78dd3,12.751624,Page-one CTR opportunity,page_one_low_ctr,CTR_REVIEW,345111,5.4,0.21,20
14090,8,content_44e481c8f55b,12.652984,Page-one CTR opportunity,page_one_low_ctr,CTR_REVIEW,312694,1.4,0.65,20
26531,9,content_cb112fce36be,12.644040,Page-one CTR opportunity,page_one_low_ctr,CTR_REVIEW,309910,5.6,0.16,104
21565,10,content_9532f197bbc8,12.641721,Page-one CTR opportunity,page_one_low_ctr,CTR_REVIEW,309192,2.0,0.87,104


## 2. Intended use and limits

### Intended use

This queue is for an SEO analyst or content editor deciding **which pages to review first**.

It is decision-support only. A high rank means that the page shows measured signals that make it worth reviewing.

### Limits

My Week-6 grouped-client validation produced:

- Base rate: about 0.511
- Precision@50: 0.660
- ROC-AUC: 0.517

The Precision@50 result suggests some useful concentration near the top of the ranking, but the ROC-AUC shows weak overall separation.

Therefore I do not treat the model or this playbook as a reliable future-prediction system.

There is also a time-window limitation: some 90-day features overlap the period used to create the decline label.

The safe claim is:

> This playbook identifies measured content patterns that may help prioritize human review. It does not prove that any recommended action will improve performance.

The decay/refresh pattern is also observational. Stale content is a review signal, not proof that refresh itself causes recovery.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
validation_summary = pd.DataFrame({
    "metric": [
        "Grouped base rate",
        "Grouped Precision@50",
        "Grouped ROC-AUC",
        "Leaky ROC-AUC test"
    ],

    "value": [
        0.511,
        0.660,
        0.517,
        0.950
    ]
})

display(validation_summary)

print("Total content rows:", len(df))
print("Actionable queue rows:", len(queue))

print(
    "Percentage sent to review:",
    f"{100 * len(queue) / len(df):.2f}%"
)

,metric,value
0,Grouped base rate,0.511
1,Grouped Precision@50,0.660
2,Grouped ROC-AUC,0.517
3,Leaky ROC-AUC test,0.950


Total content rows: 30000
Actionable queue rows: 13461
Percentage sent to review: 44.87%


## 3. Human review + the no-go list

Before acting on a recommendation, a human should check:

1. whether the page still matches the intended search intent,
2. whether the topic is seasonal,
3. whether low CTR may be caused by SERP features rather than the page,
4. whether the content is actually outdated,
5. whether another page already covers the same topic,
6. whether the suggested change fits brand and editorial requirements.

### What should NOT be automated

This playbook should not automatically:

- rewrite or publish content,
- delete pages,
- redirect pages,
- add `noindex`,
- change canonical tags,
- make revenue claims,
- make client commitments,
- treat a model score as proof that a page will decline.

Every recommendation requires human review before action.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
human_review_rules = pd.DataFrame({
    "check": [
        "Search intent",
        "Seasonality",
        "SERP context",
        "Content freshness",
        "Overlap with other pages",
        "Brand/editorial requirements"
    ],
    "required": [True] * 6
})

no_go_actions = pd.DataFrame({
    "do_not_automate": [
        "Publish or rewrite content",
        "Delete content",
        "Redirect URLs",
        "Add noindex",
        "Change canonical tags",
        "Make revenue claims",
        "Make client commitments"
    ]
})

print("Human review checklist:")
display(human_review_rules)

print("\nNo-go automation list:")
display(no_go_actions)

Human review checklist:


,check,required
0,Search intent,True
1,Seasonality,True
2,SERP context,True
3,Content freshness,True
4,Overlap with other pages,True
5,Brand/editorial requirements,True



No-go automation list:


,do_not_automate
0,Publish or rewrite content
1,Delete content
2,Redirect URLs
3,Add noindex
4,Change canonical tags
5,Make revenue claims
6,Make client commitments


## 4. Monitoring / retrain triggers

This is a light, non-production monitoring plan.

I would review the playbook about once per month.

The model or rule should be reviewed again if:

- Precision@50 falls below the current base rate.
- Precision@50 drops by more than 10 percentage points from the validated 0.66.
- More than 20% of rows lose required feature data.
- The decline-label definition changes.
- The business changes the meaning of an action or its cost.

Retraining should not happen automatically. A person should first check whether the problem is data quality, distribution change, or a real change in content behavior.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
required_features = [
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr"
]

feature_problem = (
    df[required_features].isna().any(axis=1)
    |
    (df["avg_position"] == 0)
)

feature_problem_rate = feature_problem.mean()

monitoring_rules = pd.DataFrame({
    "trigger": [
        "Precision@50 below base rate",
        "Precision@50 drops by > 0.10",
        "Required-feature issue rate > 20%",
        "Label definition changes",
        "Action/cost definition changes"
    ],

    "response": [
        "Pause and review ranking",
        "Review model and data",
        "Check data pipeline",
        "Retrain and revalidate",
        "Review playbook thresholds"
    ]
})

display(monitoring_rules)

print(
    "Current required-feature issue rate:",
    f"{feature_problem_rate:.2%}"
)

,trigger,response
0,Precision@50 below base rate,Pause and review ranking
1,Precision@50 drops by > 0.10,Review model and data
2,Required-feature issue rate > 20%,Check data pipeline
3,Label definition changes,Retrain and revalidate
4,Action/cost definition changes,Review playbook thresholds


Current required-feature issue rate: 4.02%


## 5. Exports for the paper

I export the ranked human-review queue to:

`work/outputs/w07_action_queue.csv`

I also export a small metrics JSON as the receipt for the numbers used in the paper.

The CSV contains pseudonymous IDs only. It does not contain client names, URLs, private queries, the decline label, `trend_direction`, or `trend_pct`.

I do not create a figure here because a figure is optional and the ranked queue is the main output required for the recommendations section.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
OUTPUT_DIR = REPO_ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

QUEUE_PATH = (
    OUTPUT_DIR /
    "w07_action_queue.csv"
)

METRICS_PATH = (
    OUTPUT_DIR /
    "w07_playbook_metrics.json"
)


# Only public-safe / pseudonymous fields
queue_export = queue[
    [
        "rank",
        "content_id",
        "client_id",
        "priority_score",
        "archetype",
        "reason_code",
        "action_label",
        "impressions_90d",
        "avg_position",
        "ctr",
        "days_since_last_update",
        "word_count"
    ]
].copy()

queue_export.to_csv(
    QUEUE_PATH,
    index=False
)


action_counts = {
    str(k): int(v)
    for k, v in
    queue["action_label"]
    .value_counts()
    .items()
}

metrics = {
    "grouped_base_rate": 0.511,
    "grouped_precision_at_50": 0.660,
    "grouped_roc_auc": 0.517,
    "queue_rows": int(len(queue)),
    "total_rows": int(len(df)),
    "action_counts": action_counts,
    "interpretation": "decision-support only"
}

with open(
    METRICS_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metrics,
        f,
        indent=2
    )


print("Queue saved to:")
print(QUEUE_PATH)

print("\nMetrics saved to:")
print(METRICS_PATH)

print("\nRows exported:", len(queue_export))

Queue saved to:
/content/work/outputs/w07_action_queue.csv

Metrics saved to:
/content/work/outputs/w07_playbook_metrics.json

Rows exported: 13461


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.